In [1]:
import os
import numpy as np
import pandas as pd

from astropy.time import Time
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroplan.observer import Observer

from rubin_scheduler.utils import m5_flat_sed
from rubin_scheduler.site_models import SeeingModel
from rubin_sim.skybrightness import SkyModel

# Limiting magnitude calculation for any RA/Dec/time/bandpass combination

Using the tools in rubin_scheduler and rubin_sim, and the formulas explained in [SMTN-002](https://smtn-002.lsst.io/#calculating-m5-values-in-the-lsst-operations-simulator) this notebook demonstrates how to calculate the expected limiting magnitude for any given visit. 

While the RA/Dec of a point on the sky is an important first piece of information, determining the airmass (and thus amount of atmospheric extinction, variation in delivered image quality, and variation in skybrightness) requires also specifying a time.  The m5 in all bandpasses is then shown in this notebook. 

Calculating these values for a large number of times would be computationally expensive primarily due to the skybrightness calculation; rubin_scheduler also holds pre-calculated skybrightness values in healpix grids for the time period covering the expected LSST survey. Use of the pre-calculated skybrightness is via rubin_scheduler.skybrightness_pre.SkyModelPre. For a small number of pointings, using SkyModel (as below) is more appropriate.


In [2]:
# Define observer location (for conversion to alt/az/pa + airmass)
observer = Observer.at_site("Rubin")
# Set up skybrightness model
skybrightness = SkyModel(mags=True)
# Set up the seeing model
seeing_model = SeeingModel()

In [3]:
# For the example, let's understand some convenient facts about a particular day then use those below
# Rubin DayObs
day_obs = Time("2026-01-05T12:00:00")
# What's the time at midnight? (or sun_set_time, etc.)
print(f"Sunset {observer.sun_set_time(day_obs, which='next', horizon=-12*u.deg).iso}, "\
      f"midnight {observer.midnight(day_obs, which='next').iso}, "\
      f"sunrise {observer.sun_rise_time(day_obs, which='next', horizon=-12*u.deg)}")

print(f"lunar illumination (1=full) {observer.moon_illumination(day_obs)} and phase (deg) (0=full) {np.degrees(observer.moon_phase(day_obs))}")

Sunset 2026-01-06 00:49:12.894, midnight 2026-01-06 04:48:46.130, sunrise 2461046.866893791
lunar illumination (1=full) 0.9408005228504107 and phase (deg) (0=full) 28.16389618509472 deg


In [4]:
# Define RA/Dec/Time of observation

ra = 0
dec = -40
obs_coord = SkyCoord(ra=ra*u.deg, dec=dec*u.deg)

print("ra/dec", obs_coord.ra.deg, obs_coord.dec.deg)
print("gal l/b", obs_coord.galactic.l.deg, obs_coord.galactic.b.deg)

# replace obs_time if desired
obs_time = Time("2026-01-05T01:00:00", scale='tai', format='isot')

ra/dec 0.0 -40.0
gal l/b 339.2986350396503 -73.28984903950884


In [5]:
# Check that RA/Dec is visible and calculate airmass
# Altitude < 15 degrees is inaccessible at Rubin

altaz = observer.altaz(obs_time, obs_coord)
airmass = altaz.secz.value

print(f"Altitude {altaz.alt.deg}, Azimuth {altaz.az.deg}, airmass {airmass}")

Altitude 49.73157362431672, Azimuth 242.64671885866434, airmass 1.3105736728480186


In [6]:
# Find skybrightness at this location and time (assuming target is visible)
skybrightness.set_ra_dec_mjd(lon=ra, lat=dec, mjd=obs_time.mjd, degrees=True)

# Get skybackground magnitudes
sky_bg = skybrightness.return_mags()

sky_bg

{'u': array([20.94175509]),
 'g': array([21.02896793]),
 'r': array([20.59287142]),
 'i': array([19.66836542]),
 'z': array([18.75966934]),
 'y': array([17.77855864])}

In [7]:
# Determine the fwhm in the image (depends on airmass)
atmospheric_seeing = 0.6 #arcseconds
fwhm = seeing_model(atmospheric_seeing, airmass)['fwhmEff']
fwhm = dict([(band, fw) for band, fw in zip('ugrizy', fwhm)])
fwhm

{'u': np.float64(1.061878782245665),
 'g': np.float64(1.0046995316308145),
 'r': np.float64(0.9524144488736924),
 'i': np.float64(0.916343162197038),
 'z': np.float64(0.8924076974994114),
 'y': np.float64(0.8732446329776925)}

In [8]:
# Fill these values in to calculate the m5 appropriate for these conditions
# See also SMTN-002, "Calculating m5 values in the LSST Operations Simulator"
# https://smtn-002.lsst.io/#calculating-m5-values-in-the-lsst-operations-simulator

exp_time = {'u': 38, 'g': 30, 'r': 30, 'i': 30, 'z': 30, 'y': 30}

m5_visit = {}
for band in 'ugrizy':
    m5_visit[band] = m5_flat_sed(band, musky=sky_bg[band], fwhm_eff=fwhm[band], exp_time=exp_time[band], airmass=airmass, tau_cloud=0)[0]

m5_visit

{'u': np.float64(22.915280993952624),
 'g': np.float64(24.196440062471787),
 'r': np.float64(24.043129713839793),
 'i': np.float64(23.56729372909974),
 'z': np.float64(22.975955264552017),
 'y': np.float64(21.9448403650654)}

The value above is the visit m5 value.

M5 depth for multiple images
----------------------------

When dealing with multiple images, it is possible to either approximate the depth in each image to get an approximate depth after a certain number of visits, or the actual depth in multiple visits can be combined.

In [9]:
# Let's suppose you have a string of images with variable depths

single_visit_m5s = np.array([24.2, 24.8, 24.0, 24.1, 24.3])

# Calculate approximate coadded depth
coadd_m5 = 1.25 * np.log10(np.sum(10.0 ** (0.8 * single_visit_m5s)))
print(f"From {len(single_visit_m5s)} images with depth from {single_visit_m5s.min()} to {single_visit_m5s.max()}, estimated coadded depth {coadd_m5}")

From 5 images with depth from 24.0 to 24.8, estimated coadded depth 25.23629270324802


In [10]:
# If you just have one approximate image depth and a number of images at this depth

single_visit_m5s = np.ones(10) * 24.2

coadd_m5 = 1.25 * np.log10(np.sum(10.0 ** (0.8 * single_visit_m5s)))
print(f"From {len(single_visit_m5s)} images with depth from {single_visit_m5s.min()} to {single_visit_m5s.max()}, estimated coadded depth {coadd_m5}")

From 10 images with depth from 24.2 to 24.2, estimated coadded depth 25.45


Galactic Dust Extinction
------------------------

For some purposes, knowing and including the effects of galactic extinction is important as well. 
There are many methods for calculating galactic extinction! Some even include distance (within the galaxy) as a parameter. Most of these methods rely on referencing a map to determine E(B-V) values at a given point, and then using a dust extinction law for the wavelength dependence to calculate the extinction in a given bandpass.

rubin_sim offers two different dust maps (available with the $RUBIN_SIM_DATA): 
* SFD dust maps - https://iopscience.iop.org/article/10.1086/305772 (SFD 1998)
* A 3D dust map from Will Clarkson (@willclarkson) and Alessandro Mazzi (@Thalos12) - https://github.com/willclarkson/rubinCadenceScratchWIC. This 3d dustmap should only be used for approximating dust extinction within the galaxy. 

and there are two different dust extinction laws that can be applied: 
* Cardelli, Clayton and Mathis 1989 (ApJ 345, 245) (CCM)
* O'Donnell 1994 (ApJ 422, 158) (O'Donnel)

In [11]:
from rubin_sim.maf.maps.ebv_hp import eb_vhp
from rubin_sim.maf.maps.ebv_3d_hp import ebv_3d_hp, get_x_at_nearest_y

# SFD dust map E(B-V)
ebv_sfd = eb_vhp(nside=128, ra=ra, dec=dec, interp=True)

# Clarkson & Mazzi 3d dust:  distance (pc), E(B-V)
dist_3d, ebv_3d = ebv_3d_hp(nside=128, ra=np.array([ra]), dec=np.array([dec]), interp=True)

Read map /Users/lynnej/rubin_sim_data/maps/DustMaps3D/merged_ebv3d_nside128_defaults.fits from disk


In [12]:
# The E(B-V) value returned by the SFD map is a simple number 
print(ebv_sfd)

0.08638887481601644


In [13]:
# The values return by the 3d map include distance information, so the best distance should be used
dist = 200
best_dist, ebv_dist = get_x_at_nearest_y(dist_3d, ebv_3d, dist)
print(best_dist, ebv_dist)

200.5 0.023468190448947927


In [14]:
# Calculating the magnitudes of extinction for a given source means calculating 
# A_x (the magnitudes of dust extinction in a given band)
# This can be done using some functions in rubin_sim.phot_utils.Sed

from rubin_sim.data import get_data_dir
from rubin_sim.phot_utils import Sed, Bandpass

# Pick an EBV value
ebv = ebv_sfd

dust_law = 'CCM'
#dust_law = 'ODonnell'

bandpass_dir = os.path.join(get_data_dir(), "throughputs", "baseline")
lsst_bandpass = {}
for band in 'ugrizy':
    lsst_bandpass[band] = Bandpass()
    lsst_bandpass[band].read_throughput(os.path.join(bandpass_dir, f"total_{band}.dat"))

flat_sed = Sed()
flat_sed.set_flat_sed()

if dust_law == "ODonnell":
    # for O'Donnel dust extinction
    a, b = flat_sed.setup_o_donnell_ab()
else:
    # for CCM dust extinction
    a, b = flat_sed.setup_ccm_ab()

# Calculate dust extinction - do not modify flat_sed
wavelen, flambda = flat_sed.add_dust(a, b, a_v=None, ebv=ebv, r_v=3.1, wavelen=flat_sed.wavelen, flambda=flat_sed.flambda)
dust_sed = Sed(wavelen=wavelen, flambda=flambda)

A_x = {}
for band in 'ugrizy':
    A_x[band] = dust_sed.calc_mag(lsst_bandpass[band]) - flat_sed.calc_mag(lsst_bandpass[band])

A_x

{'u': np.float64(0.412907617676062),
 'g': np.float64(0.3226227718051078),
 'r': np.float64(0.23497180493539815),
 'i': np.float64(0.17863333235638912),
 'z': np.float64(0.13779253536359803),
 'y': np.float64(0.11325925231553136)}

In [15]:
# Now correct the m5 limiting magnitude for the dust extinction (applied for either single visit or coadded depths)

m5_with_dust = {}
for band in 'ugrizy':
    m5_with_dust[band] = m5_visit[band] - A_x[band]

pd.DataFrame([m5_visit, A_x, m5_with_dust], index=['m5 visit', 'dust extinction', 'm5 with dust'])

,u,g,r,i,z,y
m5 visit,22.915281,24.196440,24.043130,23.567294,22.975955,21.944840
dust extinction,0.412908,0.322623,0.234972,0.178633,0.137793,0.113259
m5 with dust,22.502373,23.873817,23.808158,23.388660,22.838163,21.831581
